# Grokking on the Weight-Decay Clock — nonlinear test of (P2) and (P4)

Reference implementation accompanying the manuscript
*A Multi-Seed Nonlinear Test of the Weight-Decay Clock: Optimizer Dependence
and Subspace Intervention in Grokking*.

This notebook is the cleaned version of the experimental code. It reproduces
the full factorial grid of 351 runs reported in the paper, plus the three
additional controls identified in the discussion.

**Design (Table 3 of the paper)**

| Phase | Question | Runs |
|---|---|---|
| A | Does the reference benchmark reproduce? | 1 |
| B | How far do the reported values move across seeds and architectures? | 216 |
| C | Does the optimizer-dependence identity (P4) hold? | 64 |
| D | Does the intervention identity (P2) hold, and how large is the remainder? | 70 |

**Requirements.** PyTorch, NumPy, pandas, matplotlib. A CUDA device is optional:
the model has 6,528 parameters, so CPU is competitive (~360 µs/step on one core
against ~1 ms/step on an A100, where kernel-launch latency dominates). All
arithmetic is IEEE binary64 with TF32 disabled; this is not optional, since the
projector is asserted at the 1e-14 level.

**Resumability.** Every completed run is appended to `runs/runs.jsonl` and
flushed. Re-running any cell skips configurations already present.


## 1. Environment and precision policy

In [ ]:
import os, sys, json, math, time, random, platform, hashlib
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch

# --- Precision policy -------------------------------------------------
# Contraction rates of order 1e-5 per iteration and a null-space projection
# asserted at 1e-14 are not representable under TF32, and float32 rounding
# alone leaves a projection residual near 1e-7.
TRAIN_DTYPE = torch.float64
torch.set_default_dtype(TRAIN_DTYPE)
for setter in (lambda: setattr(torch.backends.cuda.matmul, "fp32_precision", "ieee"),
               lambda: setattr(torch.backends.cuda.matmul, "allow_tf32", False)):
    try:
        setter(); break
    except Exception:
        pass
try:
    torch.backends.cudnn.allow_tf32 = False
    torch.set_float32_matmul_precision("highest")
except Exception:
    pass

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ENVIRONMENT = {
    "session_start_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "python": sys.version.split()[0], "platform": platform.platform(),
    "torch": torch.__version__, "numpy": np.__version__, "pandas": pd.__version__,
    "device": str(DEVICE), "train_dtype": "float64", "tf32": "disabled",
}
if DEVICE.type == "cuda":
    _p = torch.cuda.get_device_properties(0)
    ENVIRONMENT.update(gpu_name=_p.name, gpu_capability=f"{_p.major}.{_p.minor}",
                       cuda_runtime=torch.version.cuda)

for k, v in ENVIRONMENT.items():
    print(f"{k:20s} {v}")

## 2. Storage\n\nOne JSON Lines record per run, flushed before the next run begins.

In [ ]:
BASE = os.environ.get("GROKKING_ROOT", os.path.abspath("./grokking_wd_clock"))
DIR_DATA, DIR_RUNS, DIR_ART = (os.path.join(BASE, d) for d in ("data", "runs", "artifacts"))
for d in (DIR_DATA, DIR_RUNS, DIR_ART):
    os.makedirs(d, exist_ok=True)
RUNS_JSONL = os.path.join(DIR_RUNS, "runs.jsonl")


def append_record(path: str, rec: Dict) -> None:
    with open(path, "a") as fh:
        fh.write(json.dumps(rec) + "\n")
        fh.flush(); os.fsync(fh.fileno())


def load_records(path: str) -> List[Dict]:
    out = []
    if os.path.exists(path):
        with open(path) as fh:
            for line in fh:
                line = line.strip()
                if line:
                    try: out.append(json.loads(line))
                    except json.JSONDecodeError: continue
    return out


print(f"root {BASE}\nrun log {RUNS_JSONL} ({len(load_records(RUNS_JSONL))} records)")

## 3. Configuration

Every constant that influences a reported number is declared here; nothing
downstream introduces a hidden value. This block corresponds to Table 4 of
the paper.

The reference configuration reproduces Section 7.5 of the source theory. The
initialization scale `REF_INIT_SCALE = 1.5` is **calibrated**, not quoted: the
source states a 5x inflation of standard scaling without fixing the base
convention, and that factor diverges at eta = 0.2. Because it is an inference,
`s` is carried as an explicit factor in Phase B.

In [ ]:
SMOKE_TEST = True          # <<< set False for the full grid

# ---- Reference configuration ----------------------------------------
REF_MODULUS, REF_TRAIN_FRAC, REF_WIDTH = 17, 0.55, 128
REF_BETA, REF_ETA, REF_LAMBDA = 0.9, 0.2, 3e-4
REF_INIT_SCALE, REF_DECAY = 1.5, "coupled_l2"
REF_REPORTED = {"k_mem": 4e2, "k_test50": 7e3, "k_test90": 1.05e4, "delay_ratio": 26.0}

# ---- Phase grids ------------------------------------------------------
B_SEEDS, B_MODULI, B_WIDTHS, B_INIT_SCALES = [0,1,2,3,4,5,6,7], [17,23,31], [64,128,256], [1.0,1.5,2.5]
C_SEEDS, C_BETAS, C_DECAYS = [0,1,2,3,4,5,6,7], [0.0,0.5,0.9,0.95], ["coupled_l2","decoupled_wd"]
D_SEEDS   = [0,1,2,3,4]
D_RESCALES = [0.95,0.98,0.99,0.995,0.998,1.0,1.002,1.005,1.01,1.02,1.05]
D_PROBES   = [0.5, 2.0]          # out of range by construction; never fitted
D_INTERVENE_K, D_EVAL_EVERY, D_COMPONENT_FRAC = 1000, 5, 0.5
DRIFT_TOL, SVD_TOL_REL = 1e-3, 1e-10

# ---- Training protocol ------------------------------------------------
MAX_STEPS, EVAL_EVERY = 400_000, 25
GROK_THRESHOLD, MEM_THRESHOLD, EARLY_STOP_AFTER = 0.90, 1.00, 5_000
RATE_FIT_MIN_POINTS, RATE_FIT_START_FRACTION, RATE_FIT_MIN_R2 = 40, 0.20, 0.90
BOOTSTRAP_N = 10_000

if SMOKE_TEST:
    B_SEEDS, B_MODULI, B_WIDTHS, B_INIT_SCALES = [0,1], [17], [128], [1.5]
    C_SEEDS, C_BETAS = [0,1], [0.0, 0.9]
    D_SEEDS, D_RESCALES, D_PROBES = [0,1], [0.98,0.99,1.0,1.01,1.02], [2.0]
    MAX_STEPS = 150_000


def weak_reg_ok(eta: float, lam: float, beta: float) -> bool:
    """Declared validity condition of the theory: eta*lambda < (1-sqrt(beta))^2."""
    return eta * lam < (1.0 - math.sqrt(beta)) ** 2


def bootstrap_ci(vals, n: int = None, alpha: float = 0.05, seed: int = 0):
    n = n or BOOTSTRAP_N
    v = np.asarray(vals, dtype=float); v = v[np.isfinite(v)]
    if len(v) == 0: return float("nan"), float("nan")
    if len(v) < 2:  return float(v.mean()), float(v.mean())
    rng = np.random.default_rng(seed)
    d = rng.choice(v, size=(n, len(v)), replace=True).mean(axis=1)
    return float(np.quantile(d, alpha/2)), float(np.quantile(d, 1-alpha/2))


print("Weak-regularization check, eta*lambda =", REF_ETA*REF_LAMBDA)
for b in C_BETAS:
    bound = (1-math.sqrt(b))**2
    print(f"  beta={b:<5} bound={bound:.4e}  margin={bound/(REF_ETA*REF_LAMBDA):8.1f}x  "
          f"{'OK' if weak_reg_ok(REF_ETA, REF_LAMBDA, b) else 'OUT OF SCOPE'}")

## 4. Seeding

Two seed roles are kept separate. The **split seed** fixes the data partition
and is constant throughout, so that seed-to-seed dispersion in Phase B reflects
initialization and optimization alone. The **run seed** fixes the weight
initialization and is the replication factor. Run seeds are derived by
SHA-256 of the full configuration tuple, so distinct configurations cannot
collide by arithmetic coincidence.

In [ ]:
SPLIT_SEED = 12345


def derive_seed(*parts) -> int:
    key = "|".join(str(p) for p in parts).encode()
    return int(hashlib.sha256(key).hexdigest()[:8], 16) % (2**31 - 1)


def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed % (2**32 - 1))
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


try:
    torch.use_deterministic_algorithms(True, warn_only=True)
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    DETERMINISM = "requested (warn_only=True)"
except Exception as exc:
    DETERMINISM = f"unavailable ({type(exc).__name__})"
print("determinism:", DETERMINISM)

## 5. Data

Addition modulo p with concatenated one-hot inputs and one-hot targets. The
partition is a controlled variable: drawn once under `SPLIT_SEED`, serialized,
and reused, so that it is byte-identical across library versions rather than
merely reseeded.

In [ ]:
def build_or_load_dataset(p: int, frac: float = REF_TRAIN_FRAC, seed: int = SPLIT_SEED):
    path = os.path.join(DIR_DATA, f"modadd_p{p}_frac{frac:.2f}_seed{seed}.pt")
    if os.path.exists(path):
        blob, source = torch.load(path, map_location="cpu"), "cache"
    else:
        a = torch.arange(p).repeat_interleave(p); b = torch.arange(p).repeat(p)
        c = (a + b) % p
        X = torch.zeros(p*p, 2*p); X[torch.arange(p*p), a] = 1.0; X[torch.arange(p*p), p+b] = 1.0
        Y = torch.zeros(p*p, p);   Y[torch.arange(p*p), c] = 1.0
        g = torch.Generator().manual_seed(seed)
        perm = torch.randperm(p*p, generator=g)
        n = int(frac * p * p)                      # floor: 158 of 289 at p=17
        blob = {"X_train": X[perm[:n]], "Y_train": Y[perm[:n]],
                "X_test":  X[perm[n:]], "Y_test":  Y[perm[n:]],
                "p": p, "frac": frac, "split_seed": seed,
                "n_train": int(n), "n_test": int(p*p - n)}
        torch.save(blob, path); source = "generated"
    tens = tuple(blob[k].to(DEVICE, TRAIN_DTYPE)
                 for k in ("X_train","Y_train","X_test","Y_test"))
    return tens, blob, source


DATASETS = {}
for _p in sorted(set(B_MODULI + [REF_MODULUS])):
    _t, _m, _s = build_or_load_dataset(_p)
    DATASETS[_p] = (_t, _m)
    print(f"p={_p:<3} pairs={_p**2:<5} train={_m['n_train']:<5} test={_m['n_test']:<5} [{_s}]")

assert DATASETS[17][1]["n_train"] == 158, "reference split must be 158 of 289"

## 6. Model

Bias-free two-layer perceptron with quadratic activation,
`f(x) = ((x W1)^2) W2`, parameters held as one flat vector.

Three choices are dictated by the theory and must not be varied when the
identities are under test: the **quadratic activation** makes the theory's
locally quadratic expansion exact to the order it uses; **no biases**, since
bias directions would be trivially population-active and would contaminate the
projection; and **large-norm initialization**, which places training in the
memorization-first regime.

In [ ]:
class QuadraticMLP(torch.nn.Module):
    def __init__(self, in_dim: int, width: int, out_dim: int,
                 init_scale: float = REF_INIT_SCALE):
        super().__init__()
        self.in_dim, self.width, self.out_dim = in_dim, width, out_dim
        self.n1, self.n2 = in_dim * width, width * out_dim
        self.n_params = self.n1 + self.n2
        theta = torch.cat([
            torch.randn(self.n1, device=DEVICE, dtype=TRAIN_DTYPE) * init_scale / math.sqrt(in_dim),
            torch.randn(self.n2, device=DEVICE, dtype=TRAIN_DTYPE) * init_scale / math.sqrt(width)])
        self.theta = torch.nn.Parameter(theta)

    def unpack(self, theta: Optional[torch.Tensor] = None):
        t = self.theta if theta is None else theta
        return (t[:self.n1].view(self.in_dim, self.width),
                t[self.n1:].view(self.width, self.out_dim))

    def forward(self, x: torch.Tensor, theta: Optional[torch.Tensor] = None) -> torch.Tensor:
        W1, W2 = self.unpack(theta)
        return (x @ W1).pow(2) @ W2


set_seed(0)
_m = QuadraticMLP(2*REF_MODULUS, REF_WIDTH, REF_MODULUS)
print(f"reference model: P = {_m.n_params:,}  (3pH = {3*REF_MODULUS*REF_WIDTH:,})")
del _m

## 7. Optimizers — the manipulated variable of (P4)

Identity (P4) is a statement about *where* the decay enters the momentum
recursion, so both variants are written out rather than delegated to a library
optimizer whose convention may differ between releases.

| Variant | Update |
|---|---|
| Coupled L2 | `g <- grad L + lam*theta` ; `v <- beta*v + g` ; `theta <- theta - eta*v` |
| Decoupled WD | `g <- grad L` ; `v <- beta*v + g` ; `theta <- (1-eta*lam)(theta - eta*v)` |

Note that the decoupled update shrinks the **already-updated** iterate, so it
differs from `(1-eta*lam)*theta - eta*v` by a term of order `eta^2*lam`. This
is immaterial at the operative values but is stated so the recursion can be
reimplemented without ambiguity.

In [ ]:
class HeavyBall:
    VARIANTS = ("coupled_l2", "decoupled_wd")

    def __init__(self, theta, eta, lam, beta, variant):
        assert variant in self.VARIANTS, f"unknown decay variant: {variant}"
        self.theta, self.eta, self.lam, self.beta, self.variant = theta, eta, lam, beta, variant
        self.v = torch.zeros_like(theta)

    @torch.no_grad()
    def step(self, grad):
        if self.variant == "coupled_l2":
            self.v.mul_(self.beta).add_(grad + self.lam * self.theta)
            self.theta.add_(self.v, alpha=-self.eta)
        else:
            self.v.mul_(self.beta).add_(grad)
            self.theta.add_(self.v, alpha=-self.eta)
            self.theta.add_(self.theta, alpha=-self.eta * self.lam)

    def state(self):  return {"theta": self.theta.detach().clone(), "v": self.v.clone()}

    def load(self, s):
        with torch.no_grad():
            self.theta.copy_(s["theta"]); self.v.copy_(s["v"])


def predicted_p4_ratio(beta: float) -> float:
    """(P4): T_decoupled / T_coupled = (1-beta)^-1."""
    return 1.0 / (1.0 - beta)


for b in C_BETAS:
    print(f"  beta = {b:<5} -> predicted ratio {predicted_p4_ratio(b):6.2f}x")

## 8. Training, measurement and the two rate estimators

Four quantities come out of every run: `k_mem`, `k_grok`, `delay_ratio`, and
the log-linear contraction rate of the parameter norm on the post-memorization
window.

**The parameter norm is not the theory's observable.** Phase D shows its rate
to be a factor of ~26 slower than that of the grokking component, and negative
in two thirds of grokked runs at p = 23. When a projector is supplied via
`track_U`, the component `m_k = ||U_g^T theta_k||` is tracked as well and its
own rate is fitted on the descending branch. Fits below `RATE_FIT_MIN_R2` are
flagged and retained, never silently discarded.

In [ ]:
def fit_log_linear_rate(steps, values):
    """Fit log(values) = a - rate*step. Positive rate means contraction."""
    ok = np.isfinite(values) & (values > 0)
    s, v = steps[ok], values[ok]
    if len(s) < RATE_FIT_MIN_POINTS:
        return float("nan"), float("nan"), len(s)
    y = np.log(v); A = np.vstack([s, np.ones_like(s)]).T
    coef, *_ = np.linalg.lstsq(A, y, rcond=None)
    resid = y - A @ coef
    ss_tot = float(np.sum((y - y.mean()) ** 2))
    r2 = 1.0 - float(np.sum(resid ** 2)) / ss_tot if ss_tot > 0 else float("nan")
    return float(-coef[0]), r2, len(s)


def train_run(p, width, beta, decay_variant, seed, init_scale=REF_INIT_SCALE,
              eta=REF_ETA, lam=REF_LAMBDA, max_steps=None, init_state=None,
              start_step=0, capture_state_at=None, eval_every=None,
              track_U=None, component_threshold=None, tag=""):
    max_steps  = max_steps or MAX_STEPS
    eval_every = eval_every or EVAL_EVERY
    (X, Y, Xt, Yt), _ = DATASETS[p]
    tgt_tr, tgt_te = Y.argmax(1), Yt.argmax(1)
    t0 = time.time()

    set_seed(derive_seed(seed, p, width, beta, decay_variant, init_scale, tag))
    model = QuadraticMLP(2*p, width, p, init_scale=init_scale)
    opt = HeavyBall(model.theta, eta, lam, beta, decay_variant)
    if init_state is not None:
        opt.load(init_state)

    hs, hn, ha_tr, ha_te, hc = [], [], [], [], []
    k_mem = k_test50 = k_grok = k_component = None
    captured, stop_at, step = None, None, start_step

    while step < max_steps:
        loss = torch.mean((model(X) - Y) ** 2)
        opt.step(torch.autograd.grad(loss, model.theta)[0])
        step += 1
        if capture_state_at is not None and step == capture_state_at:
            captured = opt.state()
        if step % eval_every == 0:
            with torch.no_grad():
                tr = model(X).argmax(1).eq(tgt_tr).double().mean().item()
                te = model(Xt).argmax(1).eq(tgt_te).double().mean().item()
                nrm = model.theta.norm().item()
                comp = (float((track_U.mT @ model.theta.to(track_U.dtype)).norm())
                        if track_U is not None else float("nan"))
            hs.append(step); hn.append(nrm); ha_tr.append(tr); ha_te.append(te); hc.append(comp)
            if (k_component is None and component_threshold is not None
                    and np.isfinite(comp) and comp <= component_threshold):
                k_component = step
            if k_mem is None and tr >= MEM_THRESHOLD:    k_mem = step
            if k_test50 is None and te >= 0.50:          k_test50 = step
            if k_grok is None and te >= GROK_THRESHOLD:
                k_grok = step; stop_at = step + EARLY_STOP_AFTER
            if stop_at is not None and step >= stop_at:  break
            if not math.isfinite(nrm):                   break

    steps_arr, norm_arr, comp_arr = (np.array(a, float) for a in (hs, hn, hc))

    # --- component diagnostics (non-monotone in this benchmark) ---------
    k_comp_min = comp_min = comp_growth = comp_rate = comp_rate_r2 = None
    if len(comp_arr) and np.isfinite(comp_arr).any():
        j = int(np.nanargmin(comp_arr))
        k_comp_min, comp_min = int(steps_arr[j]), float(comp_arr[j])
        comp_growth = float(comp_arr[-1] / comp_min) if comp_min > 0 else float("inf")
        w = (steps_arr >= steps_arr[0] + eval_every*5) & (steps_arr <= k_comp_min)
        if w.sum() >= 10:
            comp_rate, comp_rate_r2, _ = fit_log_linear_rate(steps_arr[w], comp_arr[w])

    # --- parameter-norm rate on the slow branch -------------------------
    hi = k_grok if k_grok is not None else (steps_arr[-1] if len(steps_arr) else 0)
    lo = k_mem if k_mem is not None else (steps_arr[0] if len(steps_arr) else 0)
    lo = max(lo, RATE_FIT_START_FRACTION * hi) if hi else lo
    w = (steps_arr >= lo) & (steps_arr <= hi)
    rate, r2, n_fit = fit_log_linear_rate(steps_arr[w], norm_arr[w])

    return {
        "tag": tag, "p": p, "width": width, "beta": beta,
        "decay_variant": decay_variant, "seed": seed, "eta": eta, "lambda": lam,
        "init_scale": init_scale,
        "k_mem": k_mem, "k_test50": k_test50, "k_grok": k_grok,
        "k_component": k_component, "component_threshold": component_threshold,
        "component_final": hc[-1] if hc else None,
        "k_component_min": k_comp_min, "component_min": comp_min,
        "component_rate": comp_rate, "component_rate_r2": comp_rate_r2,
        "component_growth_after_min": comp_growth,
        "component_monotone": (comp_growth is not None and comp_growth < 1.05),
        "delay_ratio": (k_grok/k_mem) if (k_mem and k_grok) else None,
        "grokked": k_grok is not None,
        "slow_rate": rate, "slow_rate_r2": r2, "slow_rate_points": n_fit,
        "slow_rate_ok": bool(np.isfinite(r2) and r2 >= RATE_FIT_MIN_R2),
        "final_train_acc": ha_tr[-1] if ha_tr else None,
        "final_test_acc": ha_te[-1] if ha_te else None,
        "steps_run": step, "eval_every": eval_every,
        "weak_reg_ok": weak_reg_ok(eta, lam, beta),
        "wall_clock_s": round(time.time()-t0, 1),
        "history": {"steps": hs, "theta_norm": hn, "train_acc": ha_tr,
                     "test_acc": ha_te, "component_norm": hc},
    }, captured


def run_key(r: Dict) -> Tuple:
    return (r["tag"], r["p"], r["width"], r["beta"], r["decay_variant"],
            r["seed"], r.get("init_scale", REF_INIT_SCALE))


def run_grid(cells, tag, name):
    done = {run_key(r): r for r in load_records(RUNS_JSONL)}
    pending = [c for c in cells if (tag, *c) not in done]
    print(f"{name}: {len(cells)} cells | done {len(cells)-len(pending)} | pending {len(pending)}")
    t0 = time.time()
    for i, (p, w, beta, decay, seed, sc) in enumerate(pending, 1):
        rec, _ = train_run(p, w, beta, decay, seed, init_scale=sc, tag=tag)
        append_record(RUNS_JSONL, rec); done[(tag, p, w, beta, decay, seed, sc)] = rec
        el = time.time() - t0
        print(f"  [{i}/{len(pending)}] p={p} H={w} s={sc} beta={beta} {decay:<13} "
              f"seed={seed} -> k_grok={rec['k_grok']} ({rec['wall_clock_s']:.0f}s, "
              f"eta {el/i*(len(pending)-i)/60:.0f} min)")
    return [done[(tag, *c)] for c in cells if (tag, *c) in done]


print("training and measurement defined")

## 9. The grokking-subspace projector

With `J_tr` the Jacobian of the training predictions and `J_pop` that of the
full-task predictions, the grokking subspace is

    G = ker(J_tr) ∩ ker(J_pop)^perp

A direction in G changes no training prediction to first order but does change
population predictions, leaving weight decay as the only restoring force.

The Jacobian is computed **analytically**, which is exact, faster, and
independent of framework internals. For `f(x) = ((x W1)^2) W2` with `h = x W1`:

    d f_io / d W2_jo' = h_ij^2 * delta_oo'
    d f_io / d W1_dj  = 2 * h_ij * x_id * W2_jo

Two diagnostics accompany every construction and are logically prior to any
test of (P2): the orthonormality residual and the training-Jacobian leakage
`||J_tr U_g||_inf`, which must sit at the double-precision floor.

**Cost.** The two SVDs are O(n·p·P^2) and O(p^3·P^2) with O(P^2) memory —
seconds at P = 6,528, but cubic in P and prohibitive at realistic model sizes.

In [ ]:
def analytic_jacobian(model, X, dtype=TRAIN_DTYPE):
    """Exact d f(X) / d theta, shape (n*out_dim, n_params)."""
    W1, W2 = model.unpack()
    W1, W2, Xd = W1.detach().to(dtype), W2.detach().to(dtype), X.detach().to(dtype)
    n, D = Xd.shape; H, O = W2.shape
    h = Xd @ W1
    J1 = (2.0 * torch.einsum("ij,id,jo->iodj", h, Xd, W2)).reshape(n*O, D*H)
    J2 = torch.zeros(n, O, H, O, dtype=dtype, device=Xd.device)
    idx = torch.arange(O, device=Xd.device)
    J2[:, idx, :, idx] = h.pow(2).unsqueeze(0).expand(O, n, H)
    return torch.cat([J1, J2.reshape(n*O, H*O)], dim=1)


def build_grokking_projector(model, X_train, X_full, tol_rel=SVD_TOL_REL):
    J_tr, J_pop = analytic_jacobian(model, X_train), analytic_jacobian(model, X_full)
    _, S_tr, Vh_tr = torch.linalg.svd(J_tr, full_matrices=False)
    r_tr = int((S_tr > tol_rel*float(S_tr[0])*max(J_tr.shape)).sum())
    V_r = Vh_tr[:r_tr].mT
    B = J_pop - (J_pop @ V_r) @ V_r.mT
    _, S_b, Vh_b = torch.linalg.svd(B, full_matrices=False)
    r_g = int((S_b > tol_rel*float(S_b[0])*max(B.shape)).sum())
    U_g = Vh_b[:r_g].mT
    diag = {"n_params": int(J_tr.shape[1]), "rank_J_train": r_tr,
            "null_dim": int(J_tr.shape[1] - r_tr), "grokking_dim": r_g,
            "sv_train_max": float(S_tr[0]), "sv_B_max": float(S_b[0]),
            "orthonormality_error": float((U_g.mT @ U_g -
                torch.eye(r_g, dtype=U_g.dtype, device=U_g.device)).abs().max()),
            "train_jacobian_leakage": float((J_tr @ U_g).abs().max())}
    return U_g, diag


def project(U_g, x):
    return (U_g @ (U_g.mT @ x.to(U_g.dtype))).to(x.dtype)


print("projector defined")

## 10. Phase A — reproducing the reference benchmark

Before testing anything, the benchmark must reproduce. Since the source run is
single-seed, **order-of-magnitude agreement is the criterion**, not numerical
identity. Any failure is propagated as an explicit caveat rather than absorbed.

In [ ]:
_key_A = ("A", REF_MODULUS, REF_WIDTH, REF_BETA, REF_DECAY, 0, REF_INIT_SCALE)
_existing = {run_key(r): r for r in load_records(RUNS_JSONL)}
if _key_A in _existing:
    phase_a = _existing[_key_A]; print("Phase A already complete; loaded.")
else:
    print("Phase A: reproducing the reference benchmark ...")
    phase_a, _ = train_run(REF_MODULUS, REF_WIDTH, REF_BETA, REF_DECAY, seed=0, tag="A")
    append_record(RUNS_JSONL, phase_a)

print(f"\n{'quantity':<16}{'source':>12}{'measured':>12}{'ratio':>10}")
for name, key, ref in [("k_mem","k_mem","k_mem"), ("k(test=0.5)","k_test50","k_test50"),
                       ("k(test=0.9)","k_grok","k_test90"), ("delay ratio","delay_ratio","delay_ratio")]:
    got, r = phase_a[key], REF_REPORTED[ref]
    print(f"{name:<16}{r:>12,.0f}{(f'{got:,.0f}' if got else 'n/a'):>12}"
          f"{(got/r if got else float('nan')):>10.2f}")
print(f"\nslow_rate = {phase_a['slow_rate']:.4e} /iter  (R2 = {phase_a['slow_rate_r2']:.4f}, "
      f"{phase_a['slow_rate_points']} pts, {'usable' if phase_a['slow_rate_ok'] else 'FLAGGED'})")
if not phase_a["grokked"]:
    print("\nWARNING: reference did not grok within the ceiling; downstream phases "
          "will run but their interpretation is compromised.")

## 11. Phase B — seed and architecture variability

The source states plainly that its nonlinear results are single-seed and
single-architecture. This phase converts that into a distribution and supplies
the scale against which Phase C is judged. Runs that do not reach the threshold
within `MAX_STEPS` are **right-censored**: retained with the flag, excluded from
aggregates of `k_grok` and `delay_ratio`, and counted explicitly.

In [ ]:
cells_B = [(p, w, REF_BETA, REF_DECAY, s, sc)
           for p in B_MODULI for w in B_WIDTHS for sc in B_INIT_SCALES for s in B_SEEDS]
phase_b = run_grid(cells_B, "B", "Phase B")

_COLS = ["p","width","init_scale","seed","k_mem","k_grok","delay_ratio",
         "slow_rate","slow_rate_r2","slow_rate_ok","grokked","final_test_acc"]
df_b = pd.DataFrame([{k: r[k] for k in _COLS} for r in phase_b] or None, columns=_COLS)
b_ok = df_b[df_b.grokked == True] if len(df_b) else df_b

print(f"\ngrokked: {int(df_b.grokked.sum()) if len(df_b) else 0}/{len(df_b)}")
if len(df_b):
    print(df_b.groupby("p").grokked.agg(["sum","count"]).to_string())
if len(b_ok):
    print("\ndelay ratio by cell (mean +/- sd over seeds)")
    print(b_ok.groupby(["p","width","init_scale"])["delay_ratio"]
              .agg(["mean","std","min","max","count"]).round(2).to_string())
    ref = b_ok[(b_ok.p==REF_MODULUS)&(b_ok.width==REF_WIDTH)&(b_ok.init_scale==REF_INIT_SCALE)]
    if len(ref) > 1:
        lo, hi = bootstrap_ci(ref.delay_ratio)
        print(f"\nreference cell: {ref.delay_ratio.mean():.2f} +/- {ref.delay_ratio.std():.2f}"
              f"  95% CI [{lo:.2f}, {hi:.2f}]   source reports a single value of "
              f"{REF_REPORTED['delay_ratio']:.0f}")
    neg = (b_ok.slow_rate < 0).mean()
    print(f"parameter-norm rate negative in {neg:.1%} of grokked runs "
          f"(the norm GROWS while the network generalizes)")

## 12. Phase C — the (P4) test

Paired comparison: within each `(beta, seed)` cell the two decay variants share
an identical initialization, so the ratio is formed between trajectories that
differ in exactly one factor.

Two features make this a test rather than a plausibility check. The prediction
spans 1x to 20x across the four momentum levels; and at `beta = 0` the two
update rules are algebraically identical, so that arm is a **null control**
whose scatter measures the resolution of the whole phase.

In [ ]:
cells_C = [(REF_MODULUS, REF_WIDTH, b, d, s, REF_INIT_SCALE)
           for b in C_BETAS for d in C_DECAYS for s in C_SEEDS
           if weak_reg_ok(REF_ETA, REF_LAMBDA, b)]
skipped = [b for b in C_BETAS if not weak_reg_ok(REF_ETA, REF_LAMBDA, b)]
if skipped:
    print("skipped (outside the theory's declared scope): beta =", skipped)
phase_c = run_grid(cells_C, "C", "Phase C")

_CC = ["beta","decay_variant","seed","k_mem","k_grok","slow_rate","slow_rate_r2","grokked"]
df_c = pd.DataFrame([{k: r[k] for k in _CC} for r in phase_c] or None, columns=_CC)

pairs = []
for b in (sorted(df_c.beta.unique()) if len(df_c) else []):
    for s in sorted(df_c.seed.unique()):
        cp = df_c[(df_c.beta==b)&(df_c.seed==s)&(df_c.decay_variant=="coupled_l2")]
        dc = df_c[(df_c.beta==b)&(df_c.seed==s)&(df_c.decay_variant=="decoupled_wd")]
        if len(cp) and len(dc) and cp.iloc[0].grokked and dc.iloc[0].grokked:
            pairs.append({"beta": b, "seed": s,
                          "k_coupled": cp.iloc[0].k_grok, "k_decoupled": dc.iloc[0].k_grok,
                          "ratio_k": dc.iloc[0].k_grok / cp.iloc[0].k_grok,
                          "ratio_rate": (cp.iloc[0].slow_rate / dc.iloc[0].slow_rate
                                          if dc.iloc[0].slow_rate else float("nan")),
                          "predicted": predicted_p4_ratio(b)})
df_p4 = pd.DataFrame(pairs)

rows = []
for b, g in (df_p4.groupby("beta") if len(df_p4) else []):
    pred = predicted_p4_ratio(b)
    lo_k, hi_k = bootstrap_ci(g.ratio_k); lo_r, hi_r = bootstrap_ci(g.ratio_rate)
    rows.append({"beta": b, "n_pairs": len(g), "predicted": round(pred, 3),
                 "ratio_k_mean": round(g.ratio_k.mean(), 3),
                 "ratio_k_sd": round(g.ratio_k.std(), 3),
                 "ratio_k_ci": f"[{lo_k:.2f}, {hi_k:.2f}]",
                 "k_contains_pred": bool(lo_k <= pred <= hi_k),
                 "ratio_rate_mean": round(g.ratio_rate.mean(), 3),
                 "rate_contains_pred": bool(lo_r <= pred <= hi_r),
                 "log_error_k": round(float(np.log(g.ratio_k.mean()/pred)), 3)})
df_p4_summary = pd.DataFrame(rows)
print("\nPhase C — (P4)"); print(df_p4_summary.to_string(index=False) if len(df_p4_summary) else "no pairs")

if len(df_p4_summary):
    n = df_p4_summary[df_p4_summary.beta == 0.0]
    if len(n):
        print(f"\nnull control (beta=0): {n.iloc[0].ratio_k_mean:.3f} "
              f"{n.iloc[0].ratio_k_ci} -- this is the resolution of the phase")
    print(f"prediction inside the interval at "
          f"{int(df_p4_summary.k_contains_pred.sum())}/{len(df_p4_summary)} momentum levels")
    print("\nlocalizing the deviation: the coupled arm should scale as (1-beta),")
    print("the decoupled arm should be beta-independent.")
    for b in sorted(df_c.beta.unique()):
        kc = df_c[(df_c.beta==b)&(df_c.decay_variant=="coupled_l2")].k_grok.mean()
        kd = df_c[(df_c.beta==b)&(df_c.decay_variant=="decoupled_wd")].k_grok.mean()
        print(f"  beta={b:<5} k_coup/(1-beta)={kc/(1-b):10,.0f}   k_dec={kd:10,.0f}")

## 13. Phase D — the (P2) test

Protocol: train to `D_INTERVENE_K`, snapshot `(theta, v)`, build the projector
there, rescale the projected component of **both** the parameters and the
momentum buffer by `c`, verify the training predictions are unchanged, resume.

Rescaling the parameters alone would excite the fast null root and destroy the
test.

**Grid width.** The rescalings are narrow and centred on unity. A pilot at
`c = 10` destroyed interpolation, the network re-memorized, and the measured
shift saturated in *both* directions. Two deliberately out-of-range probes are
kept, exempted from the pre-flight screen, excluded from the fit and reported
separately, because locating the breakdown is itself an object of the study.

**Two observables.** (P2) speaks about the relaxation of the grokking
component, not about test accuracy. Both are recorded; the gap between their
slopes is the measurement of how much of the component's relaxation reaches
behaviour.

In [ ]:
def run_phase_d():
    done = {(r["tag"], r["seed"], r.get("rescale_c")) for r in load_records(RUNS_JSONL)
            if r.get("tag") == "D"}
    out = []
    (X, Y, Xt, Yt), _ = DATASETS[REF_MODULUS]
    X_full = torch.cat([X, Xt], dim=0)
    all_c = [(c, False) for c in D_RESCALES] + [(c, True) for c in D_PROBES]

    for seed in D_SEEDS:
        print(f"\nseed {seed}: training to the intervention checkpoint (k0={D_INTERVENE_K})")
        base, cap = train_run(REF_MODULUS, REF_WIDTH, REF_BETA, REF_DECAY, seed,
                              capture_state_at=D_INTERVENE_K,
                              eval_every=D_EVAL_EVERY, tag="D_base")
        if cap is None:
            print("  checkpoint not reached; skipping"); continue
        append_record(RUNS_JSONL, base)

        set_seed(derive_seed(seed, REF_MODULUS, REF_WIDTH, REF_BETA, REF_DECAY,
                             REF_INIT_SCALE, "D_base"))
        model = QuadraticMLP(2*REF_MODULUS, REF_WIDTH, REF_MODULUS)
        with torch.no_grad():
            model.theta.copy_(cap["theta"])
        U_g, diag = build_grokking_projector(model, X, X_full)
        print(f"  P={diag['n_params']} rank(J_tr)={diag['rank_J_train']} "
              f"null={diag['null_dim']} dim(G)={diag['grokking_dim']}")
        print(f"  orthonormality {diag['orthonormality_error']:.2e}   "
              f"leakage {diag['train_jacobian_leakage']:.2e}")

        with torch.no_grad():
            pred_before = model(X).clone()
        g_theta, g_v = project(U_g, cap["theta"]), project(U_g, cap["v"])
        r_theta, r_v = cap["theta"] - g_theta, cap["v"] - g_v
        comp0 = float((U_g.mT @ cap["theta"].to(U_g.dtype)).norm())
        thresh = D_COMPONENT_FRAC * comp0

        # --- pre-flight drift screen: never spend a run on an uninterpretable c
        screened = []
        for c, is_probe in all_c:
            state = {"theta": r_theta + c*g_theta, "v": r_v + c*g_v}
            with torch.no_grad():
                drift = float((model(X, state["theta"]) - pred_before).abs().max())
            screened.append((c, is_probe, state, drift, drift < DRIFT_TOL))
            print(f"    c={c:<7} drift={drift:.3e}  "
                  f"{'probe' if is_probe else ('ok' if drift < DRIFT_TOL else 'OUT OF REGIME')}")

        for c, is_probe, state, drift, in_regime in screened:
            if ("D", seed, c) in done: continue
            if not in_regime and not is_probe:
                rec = {"tag":"D","p":REF_MODULUS,"width":REF_WIDTH,"beta":REF_BETA,
                       "decay_variant":REF_DECAY,"seed":seed,"init_scale":REF_INIT_SCALE,
                       "rescale_c":c,"intervene_k":D_INTERVENE_K,"train_pred_drift":drift,
                       "drift_ok":False,"is_probe":False,"skipped_untrained":True,
                       "k_grok":None,"k_component":None,"grokked":False,
                       "component_at_intervention":comp0,
                       "baseline_k_grok":base["k_grok"],"baseline_slow_rate":base["slow_rate"],
                       "projector":diag,"wall_clock_s":0.0,"eval_every":D_EVAL_EVERY}
                append_record(RUNS_JSONL, rec); out.append(rec); continue

            rec, _ = train_run(REF_MODULUS, REF_WIDTH, REF_BETA, REF_DECAY, seed,
                               init_state=state, start_step=D_INTERVENE_K,
                               eval_every=D_EVAL_EVERY, track_U=U_g,
                               component_threshold=thresh, tag="D")
            rec.update({"rescale_c": c, "intervene_k": D_INTERVENE_K,
                        "train_pred_drift": drift, "drift_ok": bool(in_regime),
                        "is_probe": bool(is_probe), "skipped_untrained": False,
                        "baseline_k_grok": base["k_grok"],
                        "baseline_slow_rate": base["slow_rate"],
                        "component_at_intervention": comp0, "projector": diag})
            append_record(RUNS_JSONL, rec); out.append(rec)
            print(f"    c={c:<7} k_comp={rec['k_component']} k_grok={rec['k_grok']} "
                  f"drift={drift:.2e}{'  [probe]' if is_probe else ''}")
    return out


phase_d = run_phase_d()
d_recs = [r for r in load_records(RUNS_JSONL) if r.get("tag") == "D"]
df_d = pd.DataFrame([{
    "seed": r["seed"], "c": r["rescale_c"], "k_grok": r["k_grok"],
    "baseline_k_grok": r["baseline_k_grok"], "slow_rate": r["baseline_slow_rate"],
    "drift": r["train_pred_drift"], "k_component": r.get("k_component"),
    "k_component_min": r.get("k_component_min"), "component_min": r.get("component_min"),
    "component_growth": r.get("component_growth_after_min"),
    "component_rate": r.get("component_rate"), "component_rate_r2": r.get("component_rate_r2"),
    "drift_ok": r["drift_ok"], "is_probe": r.get("is_probe", False),
    "skipped": r.get("skipped_untrained", False), "grokked": r["grokked"],
    "grokking_dim": r["projector"]["grokking_dim"],
    "leakage": r["projector"]["train_jacobian_leakage"]} for r in d_recs] or None)
print(f"\n{len(df_d)} Phase D records")

### 13.1 (P2) analysis

The slope is fitted on `k_component`, the theory's own observable, and
separately on `k_grok`. **What the test establishes** is not the slope — which
follows arithmetically from an exponential mode — but the *rate invariance*
behind it: whether the intervention changes the amplitude without changing the
relaxation rate. That number is printed below and is the substantive result.

In [ ]:
def fit_shift(grp, column):
    g = grp.dropna(subset=[column]).sort_values("c")
    base = g[g.c == 1.0]
    if not len(base) or len(g) < 4: return None
    x = np.log(g.c.to_numpy(float)); y = g[column].to_numpy(float) - float(base.iloc[0][column])
    if np.allclose(y, 0.0): return {"slope": 0.0, "r2": float("nan"), "n": len(g)}
    A = np.vstack([x, np.ones_like(x)]).T
    coef, *_ = np.linalg.lstsq(A, y, rcond=None)
    ss_tot = float(np.sum((y - y.mean())**2))
    r2 = 1 - float(np.sum((y - A@coef)**2))/ss_tot if ss_tot > 1e-12 else float("nan")
    return {"slope": float(coef[0]), "r2": r2, "n": len(g)}


if len(df_d):
    print(f"grokking-subspace dimension {sorted(set(int(x) for x in df_d.grokking_dim.dropna()))}"
          f"   max leakage {df_d.leakage.max():.2e}")
    print(f"drift at c=1 (must sit at the numerical floor): {df_d[df_d.c==1.0].drift.max():.2e}")

    scr = (df_d.groupby("c").agg(drift_mean=("drift","mean"), in_regime=("drift_ok","all"),
                                 probe=("is_probe","all"), trained=("skipped", lambda v: (~v).sum()))
             .reset_index())
    scr["K"] = scr.apply(lambda r: r.drift_mean/(r.c-1)**2 if r.c != 1.0 else np.nan, axis=1)
    print("\ndrift screen: delta(c) = K (c-1)^2")
    print(scr.round(6).to_string(index=False))

    mono = df_d.dropna(subset=["component_growth"])
    if len(mono):
        print(f"\nfrozen component: contracts then GROWS by "
              f"{mono.component_growth.mean():.2f}x after its minimum at step "
              f"{mono.k_component_min.mean():,.0f}; monotone in "
              f"{int(mono.component_monotone.sum()) if 'component_monotone' in mono else 0} runs")

    fit_set = df_d[df_d.drift_ok & df_d.grokked & ~df_d.is_probe & ~df_d.skipped]
    cr = fit_set.component_rate.dropna()
    if len(cr):
        theory = REF_ETA*REF_LAMBDA/(1-REF_BETA)
        print(f"\ncomponent rate {cr.mean():.4e} vs eta*lambda/(1-beta) = {theory:.4e} "
              f"-> ratio {cr.mean()/theory:.3f}")
        print(f"parameter-norm rate {fit_set.slow_rate.dropna().mean():.4e} "
              f"-> ratio {fit_set.slow_rate.dropna().mean()/theory:.3f} "
              f"(factor {cr.mean()/fit_set.slow_rate.dropna().mean():.1f} apart)")
        print(f"\nRATE INVARIANCE (the substantive claim): the fitted component rate")
        print(f"varies by {cr.std()/cr.mean()*100:.3f}% across the rescalings.")

    rows = []
    for seed, g in fit_set.groupby("seed"):
        r = g.component_rate.dropna()
        r = float(r.iloc[0]) if len(r) else float("nan")
        pred = 1.0/r if r and np.isfinite(r) and r > 0 else float("nan")
        f_c, f_m, f_a = (fit_shift(g, k) for k in ("k_component","k_component_min","k_grok"))
        if not any((f_c, f_m, f_a)): continue
        row = {"seed": seed, "component_rate": f"{r:.3e}", "predicted_slope": round(pred,1)}
        if f_c: row.update(comp_slope=round(f_c["slope"],1),
                           comp_ratio=round(f_c["slope"]/pred,4), comp_r2=round(f_c["r2"],4))
        if f_m: row.update(min_ratio=round(f_m["slope"]/pred,4))
        if f_a: row.update(acc_ratio=round(f_a["slope"]/pred,4))
        rows.append(row)
    df_p2 = pd.DataFrame(rows)
    if len(df_p2):
        print("\nPhase D — (P2)"); print(df_p2.to_string(index=False))
        for col, label in [("comp_ratio","component (theory's observable)"),
                           ("min_ratio","turning point"), ("acc_ratio","test accuracy")]:
            if col in df_p2:
                lo, hi = bootstrap_ci(df_p2[col])
                print(f"  {label:<32} {df_p2[col].mean():.4f} +/- {df_p2[col].std():.4f}"
                      f"  95% CI [{lo:.4f}, {hi:.4f}]")
else:
    df_p2 = pd.DataFrame(); print("no Phase D records")

## 14. Figures and tables

Figures are written at their final print size so that no downscaling occurs in
the manuscript and all in-figure text stays at 7 pt against 10 pt body text.
Vector PDF for submission, PNG for drafts, 600 dpi.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi":130, "savefig.dpi":600, "font.size":7,
    "font.family":"serif", "axes.grid":True, "grid.alpha":0.25, "grid.linewidth":0.4,
    "axes.spines.top":False, "axes.spines.right":False, "legend.frameon":False,
    "axes.linewidth":0.7})

def save_fig(fig, name):
    for ext in ("pdf","png"):
        fig.savefig(os.path.join(DIR_ART, f"{name}.{ext}"), bbox_inches="tight")
    plt.close(fig)

# --- Figure 1: Phase A reproduction (full width) ----------------------
h = phase_a["history"]
fig, ax = plt.subplots(1, 2, figsize=(8.3, 3.15))
ax[0].plot(h["steps"], h["train_acc"], label="train")
ax[0].plot(h["steps"], h["test_acc"], label="test")
for k, lab in [(phase_a["k_mem"], r"$k_{\mathrm{mem}}$"), (phase_a["k_grok"], r"$k_{\mathrm{grok}}$")]:
    if k:
        ax[0].axvline(k, ls=":", c="k", lw=0.9)
        ax[0].text(k, 0.05, lab, rotation=90, fontsize=6.5, va="bottom", ha="right")
ax[0].axhline(GROK_THRESHOLD, ls="--", c="grey", lw=0.8)
ax[0].set_xscale("log"); ax[0].set_ylim(-0.03, 1.03)
ax[0].set_xlabel("step $k$"); ax[0].set_ylabel("accuracy")
ax[0].legend(loc="center left"); ax[0].set_title("(a) delayed generalization", fontsize=8)
ax[1].plot(h["steps"], h["theta_norm"], c="k")
ax[1].set_xscale("log"); ax[1].set_yscale("log")
ax[1].set_xlabel("step $k$"); ax[1].set_ylabel(r"$\|\theta\|$")
ax[1].set_title(f"(b) contraction, rate {phase_a['slow_rate']:.2e} "
                f"($R^2$={phase_a['slow_rate_r2']:.3f})", fontsize=8)
fig.tight_layout(); save_fig(fig, "fig1_phase_a_reproduction")

# --- Figure 2: Phase B variability, one panel per modulus -------------
if len(b_ok):
    moduli = sorted(b_ok.p.unique())
    fig, axes = plt.subplots(1, len(moduli), figsize=(3.55*len(moduli), 3.0), sharey=True,
                             squeeze=False)
    for ax_, p in zip(axes[0], moduli):
        sub = b_ok[b_ok.p == p]
        keys = sorted({(int(r.width), float(r.init_scale)) for r in sub.itertuples()})
        data = [sub[(sub.width==w)&(sub.init_scale==s)].delay_ratio.to_numpy() for w, s in keys]
        ax_.boxplot(data, widths=0.55, showfliers=False, medianprops=dict(color="k"),
                    boxprops=dict(color="k"), whiskerprops=dict(color="k"), capprops=dict(color="k"))
        for i, d in enumerate(data, 1):
            ax_.scatter(np.full(len(d), i)+np.random.default_rng(i).uniform(-0.14,0.14,len(d)),
                        d, s=10, c="tab:blue", alpha=0.75, zorder=3)
        ax_.axhline(REF_REPORTED["delay_ratio"], ls="--", c="tab:red", lw=1.0,
                    label="value reported by the source" if p == moduli[0] else None)
        ax_.set_xticks(range(1, len(keys)+1), [f"{w}\n{s:g}" for w, s in keys], fontsize=6.5)
        ax_.set_yscale("log"); ax_.set_title(f"$p={p}$", fontsize=8.5)
        ax_.set_xlabel("width $H$ (upper) and initialization scale $s$ (lower)", fontsize=7)
    axes[0][0].set_ylabel(r"delay ratio $k_{\mathrm{grok}}/k_{\mathrm{mem}}$")
    axes[0][0].legend(loc="upper left", fontsize=6.5)
    fig.tight_layout(); save_fig(fig, "fig2_phase_b_variability")

# --- Figure 3: (P4), single column ------------------------------------
if len(df_p4):
    fig, ax = plt.subplots(figsize=(3.10, 2.95))
    lim = [0.72, max(3.0, df_p4.predicted.max()*1.7)]
    ax.plot(lim, lim, ls="--", c="0.45", lw=0.9, label=r"prediction $(1-\beta)^{-1}$")
    for b, g in df_p4.groupby("beta"):
        pred, m = predicted_p4_ratio(b), g.ratio_k.mean()
        lo, hi = bootstrap_ci(g.ratio_k)
        ax.scatter(np.full(len(g), pred), g.ratio_k, s=9, c="tab:blue", alpha=0.55, zorder=2)
        ax.errorbar([pred], [m], yerr=[[m-lo],[hi-m]], fmt="o", ms=3.6, c="k",
                    capsize=2.2, lw=0.9, zorder=4)
        ax.annotate(rf"$\beta$={b:g}", (pred, m), textcoords="offset points",
                    xytext=(6,-7), fontsize=6.2)
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlim(*lim); ax.set_ylim(*lim)
    ax.set_xlabel(r"predicted ratio $(1-\beta)^{-1}$")
    ax.set_ylabel(r"measured $k_{\mathrm{grok}}^{\mathrm{dec}}/k_{\mathrm{grok}}^{\mathrm{coup}}$")
    ax.legend(loc="upper left", fontsize=6.4)
    fig.tight_layout(pad=0.25); save_fig(fig, "fig3_phase_c_p4")

# --- Figure 4: (P2), single column, mean +/- sd -----------------------
fit_set = df_d[df_d.drift_ok & df_d.grokked & ~df_d.is_probe & ~df_d.skipped] if len(df_d) else df_d
if len(fit_set) and len(df_p2):
    sh = []
    for seed, g in fit_set.groupby("seed"):
        g = g.sort_values("c"); base = g[g.c == 1.0]
        if not len(base): continue
        for _, r in g.iterrows():
            sh.append({"c": r.c,
                       "comp": (r.k_component - base.iloc[0].k_component)
                               if pd.notna(r.k_component) else np.nan,
                       "acc": r.k_grok - base.iloc[0].k_grok})
    S = pd.DataFrame(sh).groupby("c")[["comp","acc"]].agg(["mean","std"])
    cs = S.index.to_numpy(float)
    r_g = float(pd.to_numeric(df_p2.component_rate).mean())
    fig, ax = plt.subplots(figsize=(3.10, 2.55))
    xs = np.linspace(cs.min(), cs.max(), 200)
    ax.plot(xs, np.log(xs)/r_g, ls="--", c="0.45", lw=0.9,
            label=r"prediction, slope $1/\hat r_{\mathrm{g}}$")
    ax.errorbar(cs, S[("acc","mean")], yerr=S[("acc","std")], fmt="x", ms=4, c="tab:red",
                capsize=1.8, lw=0.8, label=r"test accuracy, $k_{\mathrm{grok}}$")
    ax.errorbar(cs, S[("comp","mean")], yerr=S[("comp","std")], fmt="o", ms=3.2, c="tab:blue",
                capsize=1.8, lw=0.8, label=r"component, $k_{\mathrm{comp}}$")
    ax.set_xlabel(r"rescaling $c$ of the grokking component")
    ax.set_ylabel("shift in crossing step")
    ax.legend(loc="upper left", fontsize=6.2, labelspacing=0.3)
    fig.tight_layout(pad=0.25); save_fig(fig, "fig4_phase_d_p2")

print("figures written to", DIR_ART)
for f in sorted(os.listdir(DIR_ART)):
    if f.endswith((".pdf",".png")): print("  ", f)

In [ ]:
# --- Tables: CSV for recomputation, LaTeX for the manuscript ----------
def export_table(df, name, index=False):
    df.to_csv(os.path.join(DIR_ART, f"{name}.csv"), index=index)
    with open(os.path.join(DIR_ART, f"{name}.tex"), "w") as fh:
        fh.write(df.to_latex(index=index, escape=False))
    return name

t1 = pd.DataFrame([
    {"quantity": "k_mem",        "source": f"{REF_REPORTED['k_mem']:,.0f}",
     "measured": f"{phase_a['k_mem']:,}" if phase_a["k_mem"] else "n/a"},
    {"quantity": "k(test=0.5)",  "source": f"{REF_REPORTED['k_test50']:,.0f}",
     "measured": f"{phase_a['k_test50']:,}" if phase_a["k_test50"] else "n/a"},
    {"quantity": "k(test=0.9)",  "source": f"{REF_REPORTED['k_test90']:,.0f}",
     "measured": f"{phase_a['k_grok']:,}" if phase_a["k_grok"] else "n/a"},
    {"quantity": "delay ratio",  "source": f"{REF_REPORTED['delay_ratio']:.0f}",
     "measured": f"{phase_a['delay_ratio']:.1f}" if phase_a["delay_ratio"] else "n/a"}])
export_table(t1, "table1_reproduction")

if len(b_ok):
    t2 = (b_ok.groupby(["p","width","init_scale"])
              .agg(n=("seed","count"), delay_mean=("delay_ratio","mean"),
                   delay_sd=("delay_ratio","std"), delay_min=("delay_ratio","min"),
                   delay_max=("delay_ratio","max"), rate_mean=("slow_rate","mean"),
                   rate_sd=("slow_rate","std")).reset_index())
    for c in ("delay_mean","delay_sd","delay_min","delay_max"): t2[c] = t2[c].round(2)
    for c in ("rate_mean","rate_sd"):
        t2[c] = t2[c].map(lambda x: f"{x:.3e}" if np.isfinite(x) else "n/a")
    export_table(t2, "table2_variability")

if len(df_p4_summary): export_table(df_p4_summary, "table3_p4")
if len(df_p2):         export_table(df_p2, "table4_p2")

print("tables written:")
for f in sorted(os.listdir(DIR_ART)):
    if f.endswith((".csv",".tex")): print("  ", f)

## 15. Reproducibility manifest

A single JSON file records the environment, the full configuration, the digest
and length of the run log, the aggregate compute and the headline results, so
that any figure in the manuscript is traceable to the state that produced it.

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()


all_records = load_records(RUNS_JSONL)
MANIFEST = {
    "study": "Nonlinear test of (P2) and (P4) of the grokking weight-decay clock",
    "mode": "smoke_test" if SMOKE_TEST else "full_grid",
    "environment": ENVIRONMENT, "determinism": DETERMINISM,
    "reference_configuration": {
        "modulus": REF_MODULUS, "train_fraction": REF_TRAIN_FRAC, "width": REF_WIDTH,
        "beta": REF_BETA, "eta": REF_ETA, "lambda": REF_LAMBDA,
        "init_scale": REF_INIT_SCALE, "init_scale_calibrated": True, "decay": REF_DECAY,
        "activation": "quadratic", "loss": "mean squared error", "biases": False,
        "batching": "full batch", "split_seed": SPLIT_SEED},
    "design": {"phase_B": {"seeds": B_SEEDS, "moduli": B_MODULI, "widths": B_WIDTHS,
                            "init_scales": B_INIT_SCALES},
               "phase_C": {"seeds": C_SEEDS, "betas": C_BETAS, "decays": C_DECAYS},
               "phase_D": {"seeds": D_SEEDS, "rescalings": D_RESCALES, "probes": D_PROBES,
                            "intervention_step": D_INTERVENE_K, "eval_every": D_EVAL_EVERY,
                            "drift_tolerance": DRIFT_TOL, "svd_tol_rel": SVD_TOL_REL},
               "max_steps": MAX_STEPS, "eval_every": EVAL_EVERY,
               "grok_threshold": GROK_THRESHOLD, "mem_threshold": MEM_THRESHOLD},
    "integrity": {"runs_jsonl_sha256": sha256_file(RUNS_JSONL) if os.path.exists(RUNS_JSONL) else None,
                  "runs_jsonl_bytes": os.path.getsize(RUNS_JSONL) if os.path.exists(RUNS_JSONL) else 0,
                  "n_records": len(all_records)},
    "compute": {"total_accelerator_hours": round(sum(r.get("wall_clock_s", 0)
                                                     for r in all_records)/3600, 3),
                "runs": len(all_records)},
    "headline_results": {
        "phase_A_reproduced": bool(phase_a.get("grokked")),
        "phase_A_delay_ratio": phase_a.get("delay_ratio"),
        "phase_B_grok_rate": float(df_b.grokked.mean()) if len(df_b) else None,
        "phase_C_levels_supporting_P4": int(df_p4_summary.k_contains_pred.sum())
            if len(df_p4_summary) else 0,
        "phase_C_levels_tested": int(len(df_p4_summary)),
        "phase_D_component_slope_ratio": float(df_p2.comp_ratio.mean())
            if len(df_p2) and "comp_ratio" in df_p2 else None,
        "phase_D_accuracy_slope_ratio": float(df_p2.acc_ratio.mean())
            if len(df_p2) and "acc_ratio" in df_p2 else None,
        "phase_D_max_train_pred_drift": float(df_d.drift.max()) if len(df_d) else None},
}
with open(os.path.join(DIR_ART, "manifest.json"), "w") as fh:
    json.dump(MANIFEST, fh, indent=2, default=str)

print(f"runs {len(all_records)}   compute {MANIFEST['compute']['total_accelerator_hours']:.2f} h")
print("manifest ->", os.path.join(DIR_ART, "manifest.json"))

## 16. Additional controls (optional)

These are the follow-up experiments identified in the discussion. They are not
part of the 351-run grid and are written under distinct tags so that nothing
already recorded is overwritten.

| Control | Question it answers | Cost |
|---|---|---|
| C2 | Is the (P4) deviation conditional on the calibrated init scale? | ~3.1 GPU-h |
| C3 | Does the rate ratio survive on the component rather than on the norm? | ~1.6 GPU-h |
| B31 | Is the p = 31 censoring a budget artefact or a regime change? | ~9.6 GPU-h |

Note that C2 and C3 must be run in the **same numerical environment** as the
main grid. Mixing CPU and GPU results violates the discipline this study
advocates, since CPU-versus-GPU execution is known to flip individual seed
outcomes.

In [ ]:
RUN_ADDITIONAL_CONTROLS = False      # <<< set True to execute

if RUN_ADDITIONAL_CONTROLS:
    # ---- C2: (P4) at the remaining initialization scales --------------
    cells_C2 = [(REF_MODULUS, REF_WIDTH, b, d, s, sc)
                for sc in [1.0, 2.5] for b in C_BETAS for d in C_DECAYS for s in C_SEEDS
                if weak_reg_ok(REF_ETA, REF_LAMBDA, b)]
    run_grid(cells_C2, "C2", "Phase C2 (init-scale robustness)")

    # ---- C3: (P4) with the component tracked --------------------------
    (X, Y, Xt, Yt), _ = DATASETS[REF_MODULUS]
    X_full = torch.cat([X, Xt], dim=0)
    done_c3 = {(r["beta"], r["decay_variant"], r["seed"]) for r in load_records(RUNS_JSONL)
               if r.get("tag") == "C3"}
    for beta in C_BETAS:
        if not weak_reg_ok(REF_ETA, REF_LAMBDA, beta): continue
        for seed in C_SEEDS:
            # one projector per (beta, seed), built on the coupled arm and reused
            # for both arms, so the ratio stays a like-for-like comparison
            _, cap = train_run(REF_MODULUS, REF_WIDTH, beta, "coupled_l2", seed,
                               capture_state_at=D_INTERVENE_K,
                               eval_every=D_EVAL_EVERY, tag="C3_ref")
            if cap is None: continue
            set_seed(derive_seed(seed, REF_MODULUS, REF_WIDTH, beta, "coupled_l2",
                                 REF_INIT_SCALE, "C3_ref"))
            m = QuadraticMLP(2*REF_MODULUS, REF_WIDTH, REF_MODULUS)
            with torch.no_grad(): m.theta.copy_(cap["theta"])
            U_g, diag = build_grokking_projector(m, X, X_full)
            comp0 = float((U_g.mT @ cap["theta"].to(U_g.dtype)).norm())
            for decay in C_DECAYS:
                if (beta, decay, seed) in done_c3: continue
                rec, _ = train_run(REF_MODULUS, REF_WIDTH, beta, decay, seed,
                                   track_U=U_g, component_threshold=D_COMPONENT_FRAC*comp0,
                                   eval_every=D_EVAL_EVERY, tag="C3")
                rec.update({"projector": diag, "component_at_reference": comp0})
                append_record(RUNS_JSONL, rec)
                print(f"  beta={beta:<5} {decay:<13} seed={seed} "
                      f"k_grok={rec['k_grok']} component_rate={rec['component_rate']}")

    # ---- B31: raised ceiling at p = 31 --------------------------------
    MAX_STEPS_P31 = 800_000
    done_b31 = {(r["width"], r["init_scale"], r["seed"]) for r in load_records(RUNS_JSONL)
                if r.get("tag") == "B31"}
    for w in B_WIDTHS:
        for sc in B_INIT_SCALES:
            for s in B_SEEDS:
                if (w, sc, s) in done_b31: continue
                rec, _ = train_run(31, w, REF_BETA, REF_DECAY, s, init_scale=sc,
                                   max_steps=MAX_STEPS_P31, tag="B31")
                append_record(RUNS_JSONL, rec)
                print(f"  p=31 H={w} s={sc} seed={s} -> grokked={rec['grokked']} "
                      f"k_grok={rec['k_grok']} final_test_acc={rec['final_test_acc']:.3f}")
else:
    print("additional controls disabled; set RUN_ADDITIONAL_CONTROLS = True to run them")